# Evaluation Rubric

Deterministic scoring criteria for evaluating model-generated product descriptions.
Each criterion is rated as good, ok, or bad using explicit measurable rules.


## Criterion Definitions

### 1. Fluency (Natural, easy-to-read sentences)

good:
Sentences read naturally with smooth flow and no awkward phrasing.

ok:
Minor awkward phrasing but overall understandable and readable.

bad:
Multiple unnatural or difficult-to-follow sentences that disrupt readability.


### 2. Grammar (Correct spelling & punctuation)

good:
No spelling or punctuation errors.

ok:
1 minor grammar/spelling/punctuation error.

bad:
2 or more grammar/spelling/punctuation errors.


### 3. Tone (Friendly, credible sales voice)

good:
Consistently friendly, engaging, and credible sales tone throughout.

ok:
Mostly appropriate tone with minor inconsistency OR slightly neutral wording.

bad:
Tone inappropriate, overly formal, robotic, or not aligned with sales voice.


### 4. Length (50–90 words)

good:
50–90 words

ok:
40–49 words OR 91–110 words

bad:
Less than 40 words OR more than 110 words


### 5. Grounding (The response contains information included only in the context. The response does not reference any outside information.)

good:
All claims strictly supported by provided information.

ok:
One minor unsupported detail added.

bad:
Multiple unsupported claims OR contradicts provided information.


### 6. Latency (Average time per call)

good:
Response time within acceptable production threshold (e.g., ≤2 seconds).

ok:
Response time slightly above threshold (2–4 seconds).

bad:
Response time exceeds acceptable limit (>4 seconds).


### 7. Cost (Average price per call per 1K tokens)

good:
Within target budget threshold.

ok:
Slightly above target budget (≤25% over).

bad:
More than 25% above target budget threshold.

## Pass / Fail Definition

### Cumulative pass rule

A response passes if:

- At least 4 criteria are rated good
- No more than 1 criterion is rated bad


### Automatic failure rules (go / no-go)

A response automatically fails if:

- Grounding is not good
OR
- Grammar is bad

# LLM Product description generation

In [10]:
system_prompt = """
You are a professional ecommerce product description writer.

Write concise, engaging product descriptions using:

- product name
- key features
- material
- warranty

Follow these rules:

1. Length: 40–60 words
2. Tone: professional and persuasive
3. Highlight 2–3 key features
4. Mention material if relevant
5. Mention warranty at the end
6. Do NOT invent information
"""

In [11]:
%pip install pandas openpyxl


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd

df = pd.read_excel("Assignment_01_product_dataset.xlsx")

In [13]:
def format_product(row):
    return f"""
Product name: {row['product_name']}
Attributes: {row['Product_attribute_list']}
Material: {row['material']}
Warranty: {row['warranty']}
"""



In [14]:
format_product(df.iloc[0])

'\nProduct name: Apple iPhone 15 Pro\nAttributes: features: A17 Pro chip, 120\u202fHz ProMotion display, USB‑C fast charging; dimensions: compact\nMaterial: titanium frame, Ceramic Shield glass\nWarranty: 1‑year limited warranty\n'

In [15]:
%pip install python-dotenv openai



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [16]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY")
)



In [39]:
import time

results = []

for _, row in df.iterrows():

    product_prompt = format_product(row)

    start_time = time.time()

    response = client.chat.completions.create(
        model="meta-llama/Meta-Llama-3.1-8B-Instruct",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": product_prompt}
        ],
        temperature=0,
        top_p=0.1
    )

    end_time = time.time()

    latency_ms = (end_time - start_time) * 1000

    results.append({
        "description": response.choices[0].message.content,
        "latency_ms": latency_ms,
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens
    })
    
metrics_df = pd.DataFrame(results)


In [40]:
metrics_df.head()

,description,latency_ms,input_tokens,output_tokens
0,"""Experience unparalleled performance with the ...",2225.660324,153,63
1,"""Unlock unparalleled mobile photography with t...",2221.701622,156,83
2,"""Capture life's moments with precision and cla...",2031.871796,154,72
3,"""Immerse yourself in pure sound with the Sony ...",2500.877380,153,71
4,"""Immerse yourself in exceptional sound with th...",2290.181875,147,75


In [19]:
df = pd.concat([df, metrics_df], axis=1)

In [21]:
rubric_columns = [
    "fluency_score",
    "grammar_score",
    "tone_score",
    "length_score",
    "grounding_score",
    "latency_score",
    "cost_score"
]

for col in rubric_columns:
    df[col] = ""

In [25]:
df["final_score"] = ""


In [26]:
df.to_excel("assignment_01.xlsx", index=False)

# LLM Judge that grades product descriptions

In [41]:
judge_prompt = """\
# Instruction
You are an expert evaluator. Your task is to evaluate the quality of the responses generated by an AI model.
You'll be provided with the input and an AI-generated responses.
You should first read the input carefully for analyzing the task, and then evaluate the quality of the responses based on the Criteria provided in the Evaluation section below.
You will assign the response a rating following the Rating Rubric and Evaluation Steps and only choose ratings from the Rating Rubric.

# Evaluation
## Metric Definition
You will be assessing product description generation quality.

## Criteria
Fluency: The response is well-organized and easy to read.

## Rating Rubric
3: (good). Sentences read naturally with smooth flow and no awkward phrasing.
2: (ok). Minor awkward phrasing but overall understandable and readable.
1: (bad). Multiple unnatural or difficult-to-follow sentences that disrupt readability.

## Criteria
Grammar: Correct spelling & punctuation.

## Rating Rubric
3: (good). No spelling or punctuation errors.
2: (ok). 1 minor grammar/spelling/punctuation error.
1: (bad). 2 or more grammar/spelling/punctuation errors.

## Criteria
Tone: Friendly, credible sales voice.

## Rating Rubric
3: (good). Consistently friendly, engaging, and credible sales tone throughout.
2: (ok). Mostly appropriate tone with minor inconsistency OR slightly neutral wording.
1: (bad). Tone inappropriate, overly formal, robotic, or not aligned with sales voice.

## Criteria
Length: 50–90 words

## Rating Rubric
3: (good). 50–90 words
2: (ok). 40–49 words OR 91–110 words
1: (bad). Less than 40 words OR more than 110 words

## Criteria
Grounding: The response contains information included only in the context. The response does not reference any outside information.
## Rating Rubric
3: (good). All claims strictly supported by provided information.
2: (ok). One minor unsupported detail added.
1: (bad). Multiple unsupported claims OR contradicts provided information.

## Evaluation Steps
STEP 1: Assess the response in aspects of instruction following, fluency, grammar, tone, length and grounding according to the criteria.
STEP 2: Score based on the rubric.

# User Inputs and AI-generated Response
## User Inputs

### Prompt
{product_prompt}

"""

In [ ]:
evaluation_results = []

for _, row in metrics_df.iterrows():

    description = row["description"]

    judge_response = client.chat.completions.create(
        model="meta-llama/Meta-Llama-3.1-8B-Instruct",
        messages=[
            {"role": "system", "content": judge_prompt},
            {"role": "user", "content": description}
        ],
        temperature=0
    )

    evaluation_results.append(
        judge_response.choices[0].message.content
    )